# RSNA Knee Abnormality Detection — standalone submission notebook

**Pipeline (identical to EXP-004 / EXP-008 / EXP-011):** frozen DINOv2-Base CLS embeddings of 9 slices per study, mean-pooled, then a per-target `StandardScaler + LogisticRegression(C=1, class_weight="balanced")` trained on the 58 fully-labelled training studies. Test reports are never used.

**What is different from the experiment notebook**
- Self-contained: nothing is read from `/kaggle/working`, GitHub, or the internet. Training features are recomputed here (about 30 s).
- Never crashes on a bad study: series fall back sagittal-fluid → coronal-fluid → any sagittal → any coronal → axial-fluid → any series; slice ordering and sampling tolerate missing/duplicate `InstanceNumber` and short series; unreadable studies get the training prior.
- A valid prior-only `submission.csv` is written first, so a file exists before inference. Any pre-existing submission is backed up before replacement. The final file is validated; invalid model output leaves the prior-only file in place.
- Everything is logged to `run_report.json`.

**Notebook settings:** Internet **OFF**, Accelerator **GPU ON**, inputs = the competition + the DINOv2 model (`metaresearch/dinov2`, PyTorch, base).

**Before you submit:**
1. Set `SIMULATE_N = 100`, run the notebook, and read the report: fallback rates and seconds per study.
2. Set `SIMULATE_N = 0`.
3. **Save Version → Save & Run All**, open the version's output, and check `run_report.json` says `status: OK`.
4. Only then submit from the version page.

In [1]:
import gc, json, os, shutil, sys, time, traceback, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ------------------------------ settings ------------------------------
DATA = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
MODEL_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/base/1"
WORK = Path("/kaggle/working")
SUBMISSION_PATH = WORK / "submission.csv"

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
N_SLICES = 9
# (0.0, 1.0) reproduces EXP-004/008/011 exactly (first and last slice included).
# Example for central slices only: (0.15, 0.85). Training and test use the same setting.
SLICE_RANGE = (0.0, 1.0)
SEED = 42

SIMULATE_N = 100                    # >0 = dry run on this many unlabelled TRAIN studies; MUST be 0 to submit
TIME_BUDGET_SECONDS = 7.5 * 3600  # stop extracting and use the prior for the rest (competition limit is 9 h)
MAX_SERIES_TRIES = 4              # series attempted per study before giving up on it
MIN_TRAIN_STUDIES = 50            # fewer extracted gold studies than this -> try an attached feature cache
MIN_MODEL_STUDIES = 20            # fewer than this -> prior-only submission
HIGH_FAILURE_RATE = 0.20          # status becomes WARN if more test studies than this fall back to the prior

warnings.filterwarnings("ignore")
START = time.time()
LOG = []   # one dict per fallback / failure event; summarised in run_report.json

print("pydicom", pydicom.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
print("SIMULATE_N =", SIMULATE_N, "(must be 0 for the real submission)" if SIMULATE_N else "(real submission mode)")

pydicom 3.0.2 | numpy 2.1.3 | pandas 2.3.3
SIMULATE_N = 100 (must be 0 for the real submission)


In [2]:
def log(event, **kw):
    LOG.append({"event": event, "t": round(time.time() - START, 1), **kw})


def elapsed():
    return time.time() - START


# ------------------------------ model ------------------------------
def find_model_dir():
    p = Path(MODEL_PATH)
    if (p / "config.json").exists():
        return str(p)
    # Fall back to any attached DINOv2 model (only under /kaggle/input/models, never the DICOM folders).
    for cfg in sorted(Path("/kaggle/input/models").glob("**/config.json")):
        try:
            if "dinov2" in cfg.read_text().lower():
                log("model_path_fallback", path=str(cfg.parent))
                return str(cfg.parent)
        except Exception:
            pass
    raise FileNotFoundError(f"DINOv2 weights not found at {MODEL_PATH} or under /kaggle/input/models")


def load_encoder():
    """Same extraction as EXP-004: CLS token of last_hidden_state, 768-d per slice."""
    import torch
    from transformers import AutoImageProcessor, AutoModel

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    path = find_model_dir()
    processor = AutoImageProcessor.from_pretrained(path, local_files_only=True)
    model = AutoModel.from_pretrained(path, local_files_only=True).to(device).eval()

    def encode(images):
        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model(**inputs)
        return out.last_hidden_state[:, 0, :].float().cpu().numpy()

    return encode, str(device)


# ------------------------------ series choice ------------------------------
def ranked_series(series_df, study_id):
    """Ordered list of (series_id, rung). The first entry matches the EXP-004 choice when a fluid sagittal exists."""
    c = series_df[series_df["StudyInstanceUID"].astype(str) == str(study_id)]
    if c.empty:
        return []
    plane = (c["Anatomical_Plane"].astype(str).str.strip().str.lower()
             if "Anatomical_Plane" in c else pd.Series("", index=c.index))
    fluid = ((pd.to_numeric(c["Fluid_Sensitive"], errors="coerce") == 1)
             if "Fluid_Sensitive" in c else pd.Series(False, index=c.index))
    everything = pd.Series(True, index=c.index)
    rungs = [
        ("sagittal_fluid", (plane == "sagittal") & fluid),
        ("coronal_fluid", (plane == "coronal") & fluid),
        ("sagittal_any", plane == "sagittal"),
        ("coronal_any", plane == "coronal"),
        ("axial_fluid", (plane == "axial") & fluid),
        ("any_series", everything),
    ]
    out, seen = [], set()
    for name, mask in rungs:
        for sid in sorted(c.loc[mask, "SeriesInstanceUID"].astype(str)):
            if sid not in seen:
                seen.add(sid)
                out.append((sid, name))
    return out


# ------------------------------ DICOM ------------------------------
def ordered_dicom_paths(folder):
    """Sort slices by InstanceNumber; fall back to position along the slice normal, then to file name."""
    paths = sorted(Path(folder).glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM files in {folder}")
    heads = []
    for p in paths:
        try:
            heads.append((p, pydicom.dcmread(p, stop_before_pixels=True)))
        except Exception:
            log("header_unreadable", file=str(p))
    if not heads:
        raise ValueError(f"No readable DICOM headers in {folder}")

    inst = [getattr(ds, "InstanceNumber", None) for _, ds in heads]
    if all(i is not None for i in inst) and len({int(i) for i in inst}) == len(inst):
        order = sorted(range(len(heads)), key=lambda k: (int(inst[k]), heads[k][0].name))
        return [heads[k][0] for k in order], "instance_number"
    try:
        iop = np.array(heads[0][1].ImageOrientationPatient, dtype=float)
        normal = np.cross(iop[:3], iop[3:])
        pos = [float(np.dot(normal, np.array(ds.ImagePositionPatient, dtype=float))) for _, ds in heads]
        order = sorted(range(len(heads)), key=lambda k: (pos[k], heads[k][0].name))
        return [heads[k][0] for k in order], "image_position"
    except Exception:
        return sorted(h[0] for h in heads), "filename"


def dicom_to_image(path):
    """Identical preprocessing to EXP-004."""
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2:
        raise ValueError(f"Unsupported pixel array shape {pixels.shape}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1))
    pixels += float(getattr(ds, "RescaleIntercept", 0))
    lo, hi = np.percentile(pixels, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        raise ValueError(f"Invalid intensity range: {path}")
    pixels = np.clip((pixels - lo) / (hi - lo), 0, 1)
    pixels = (pixels * 255).astype(np.uint8)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = 255 - pixels
    return Image.fromarray(pixels).convert("RGB")


def sample_indices(n):
    lo, hi = SLICE_RANGE
    return np.rint(np.linspace(lo * (n - 1), hi * (n - 1), N_SLICES)).astype(int)


def extract_study(encode, series_df, root, study_id):
    """Return ((9, 768) features, info). Tries up to MAX_SERIES_TRIES series; raises only if all fail."""
    tries = ranked_series(series_df, study_id)[:MAX_SERIES_TRIES]
    if not tries:
        raise RuntimeError("study has no rows in the series table")
    last_err = None
    for rank, (sid, rung) in enumerate(tries):
        try:
            paths, order_mode = ordered_dicom_paths(Path(root) / str(study_id) / sid)
            idx = sample_indices(len(paths)).tolist()
            images, failed = {}, 0
            for k in sorted(set(idx)):
                try:
                    images[k] = dicom_to_image(paths[k])
                except Exception as e:
                    failed += 1
                    log("slice_failed", study=str(study_id), series=sid, slice=k, error=repr(e)[:160])
            if not images:
                raise RuntimeError("no readable sampled slices")
            readable = sorted(images)
            # An unreadable slice is replaced by its nearest readable neighbour.
            batch = [images[min(readable, key=lambda j: abs(j - k))] for k in idx]
            feats = np.asarray(encode(batch), dtype=np.float32)
            if feats.shape != (N_SLICES, 768) or not np.isfinite(feats).all():
                raise RuntimeError(f"invalid features {feats.shape}")
            info = {"series_id": sid, "rung": rung, "rung_rank": rank, "order_mode": order_mode,
                    "n_slices": len(paths), "unique_slices_sampled": len(set(idx)),
                    "unreadable_slices": failed}
            return feats, info
        except Exception as e:
            last_err = e
            log("series_failed", study=str(study_id), series=sid, rung=rung, error=repr(e)[:200])
    raise RuntimeError(f"all {len(tries)} candidate series failed; last error: {last_err!r}")

In [3]:
def resolve_inputs():
    """Work out which studies to predict, which columns to write, and where their DICOMs live."""
    if SIMULATE_N > 0:
        labels = pd.read_csv(DATA / "train.csv")
        gold_ids = set(labels.dropna(subset=TARGETS)["StudyInstanceUID"].astype(str))
        pool = sorted(s for s in labels["StudyInstanceUID"].astype(str) if s not in gold_ids)
        return dict(ids=pool[:SIMULATE_N], id_col="StudyInstanceUID",
                    columns=["StudyInstanceUID"] + TARGETS, root=DATA / "train_series",
                    series=pd.read_csv(DATA / "train_series.csv"),
                    out=WORK / "simulated_submission.csv", simulated=True)

    test_ids = pd.read_csv(DATA / "test.csv")["StudyInstanceUID"].astype(str).tolist()
    if not test_ids:
        raise ValueError("test.csv contains no study IDs")
    if len(set(test_ids)) != len(test_ids):
        raise ValueError("test.csv contains duplicate StudyInstanceUID values")

    # The official test.csv is authoritative for IDs and row order. The sample
    # submission may define column order, but must never replace the test IDs.
    columns = ["StudyInstanceUID"] + TARGETS
    sample_path = DATA / "sample_submission.csv"
    if sample_path.exists():
        sample = pd.read_csv(sample_path)
        if len(sample.columns) > 1:
            sample_columns = sample.columns.tolist()
            if sample_columns[0] != "StudyInstanceUID" or set(sample_columns[1:]) != set(TARGETS) or len(sample_columns) != len(TARGETS) + 1:
                raise ValueError(f"Unexpected sample_submission schema: {sample_columns}")
            columns = sample_columns
            sample_ids = sample["StudyInstanceUID"].astype(str).tolist()
            if sample_ids != test_ids:
                log("sample_vs_test_id_mismatch", n_sample=len(sample_ids), n_test=len(test_ids),
                    action="using test.csv IDs and order")

    return dict(ids=test_ids, id_col="StudyInstanceUID", columns=columns, root=DATA / "test_series",
                series=pd.read_csv(DATA / "test_series.csv"), out=SUBMISSION_PATH, simulated=False)


def prior_table(inputs, prior):
    table = pd.DataFrame({inputs["id_col"]: inputs["ids"]})
    for col in inputs["columns"][1:]:
        table[col] = float(prior.get(col, 0.5))
    return table[inputs["columns"]]


def build_training_set(encode, state):
    """Recompute features for the fully-labelled studies; fall back to an attached cache if that fails."""
    labels = pd.read_csv(DATA / "train.csv")
    series = pd.read_csv(DATA / "train_series.csv")
    gold = labels.dropna(subset=TARGETS).sort_values("StudyInstanceUID").reset_index(drop=True)
    if len(gold) != 58:
        log("unexpected_gold_count", n=len(gold))
    feats, kept = [], []
    for _, row in gold.iterrows():
        sid = str(row["StudyInstanceUID"])
        try:
            f, info = extract_study(encode, series, DATA / "train_series", sid)
            feats.append(f.mean(axis=0))
            kept.append(sid)
            state["train_rungs"][info["rung"]] += 1
        except Exception as e:
            log("train_study_failed", study=sid, error=repr(e)[:200])
    print(f"Training features: {len(kept)}/{len(gold)} gold studies in {elapsed():.0f}s")
    X, gold_kept = (np.stack(feats) if feats else np.zeros((0, 768), np.float32)), gold[gold["StudyInstanceUID"].astype(str).isin(kept)]

    if len(kept) < MIN_TRAIN_STUDIES:
        # Look for an attached copy of the EXP-004 cache (features.npy + study_ids.npy + labels.csv).
        for feat_path in sorted(Path("/kaggle/input").glob("*/features.npy")) + sorted(Path("/kaggle/input").glob("*/*/features.npy")):
            d = feat_path.parent
            try:
                f3 = np.load(d / "features.npy")
                ids = np.load(d / "study_ids.npy", allow_pickle=True).astype(str)
                lab = pd.read_csv(d / "labels.csv")
                if f3.ndim == 3 and f3.shape[1:] == (N_SLICES, 768) and len(ids) == len(f3) == len(lab):
                    lab = lab.assign(StudyInstanceUID=lab["StudyInstanceUID"].astype(str))
                    if lab["StudyInstanceUID"].tolist() == ids.tolist() and lab[TARGETS].notna().all().all():
                        log("train_cache_fallback", path=str(d), n=len(ids))
                        state["notes"].append(f"training features loaded from attached cache {d}")
                        return f3.mean(axis=1).astype(np.float32), lab
            except Exception as e:
                log("train_cache_unusable", path=str(d), error=repr(e)[:160])
    return X, gold_kept.reset_index(drop=True)


def fit_models(X, gold):
    models = {}
    for t in TARGETS:
        y = pd.to_numeric(gold[t], errors="coerce").to_numpy()
        ok = np.isfinite(y)
        if ok.sum() < 10 or len(np.unique(y[ok])) < 2:
            models[t] = None
            log("target_without_model", target=t, n=int(ok.sum()))
            continue
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=SEED))
        clf.fit(X[ok], y[ok].astype(int))
        models[t] = clf
    return models


def validate_submission(sub, columns, ids):
    problems = []
    if sub.columns.tolist() != columns:
        problems.append("columns differ from the expected list")
    if len(sub) != len(ids):
        problems.append(f"row count {len(sub)} != {len(ids)}")
    elif sub[columns[0]].astype(str).tolist() != list(ids):
        problems.append("study id order differs")
    vals = sub[columns[1:]].to_numpy(dtype=np.float64)
    if not np.isfinite(vals).all():
        problems.append("NaN or infinite predictions")
    elif (vals < 0).any() or (vals > 1).any():
        problems.append("predictions outside [0, 1]")
    return problems


def run_pipeline():
    """Never raises. A prior-only submission is written first and only replaced by a validated model submission."""
    state = {"status": "OK", "notes": [], "device": None, "train_rungs": Counter(), "test_rungs": Counter(),
             "order_modes": Counter(), "n_test": 0, "n_model": 0, "n_prior_filled": 0, "seconds_per_study": None}
    prior, inputs = {}, None
    try:
        inputs = resolve_inputs()
        state["n_test"] = len(inputs["ids"])
        state["simulated"] = inputs["simulated"]
        try:
            lab = pd.read_csv(DATA / "train.csv").dropna(subset=TARGETS)
            prior = {t: float(pd.to_numeric(lab[t]).mean()) for t in TARGETS}
        except Exception as e:
            log("prior_unavailable", error=repr(e)[:200])
        # Preserve any pre-existing submission before creating the safety-net file.
        if inputs["out"].exists():
            backup = WORK / f"submission.csv.backup.{time.strftime('%Y%m%d_%H%M%S')}"
            shutil.copy2(inputs["out"], backup)
            log("existing_submission_backed_up", path=str(backup))
        prior_table(inputs, prior).to_csv(inputs["out"], index=False)
        print(f"[step 0] prior-only safety-net file written: {inputs['out']} ({len(inputs['ids'])} studies)")

        encode, state["device"] = load_encoder()
        print("[step 1] encoder loaded on", state["device"])
        if state["device"] == "cpu":
            state["notes"].append("running on CPU; extraction will be slow")
        X_train, gold = build_training_set(encode, state)
        if len(X_train) < MIN_MODEL_STUDIES:
            raise RuntimeError(f"only {len(X_train)} usable training studies; keeping prior-only submission")
        models = fit_models(X_train, gold)
        print(f"[step 2] fitted {sum(m is not None for m in models.values())}/{len(TARGETS)} target models on {len(X_train)} studies")

        feats, ok_ids, cache = [], [], {}
        t0 = time.time()
        for i, sid in enumerate(inputs["ids"], start=1):
            if elapsed() > TIME_BUDGET_SECONDS:
                log("time_budget_exceeded", processed=i - 1, remaining=len(inputs["ids"]) - i + 1)
                state["notes"].append("time budget exceeded; remaining studies use the prior")
                break
            if sid not in cache:
                try:
                    f, info = extract_study(encode, inputs["series"], inputs["root"], sid)
                    cache[sid] = f.mean(axis=0)
                    state["test_rungs"][info["rung"]] += 1
                    state["order_modes"][info["order_mode"]] += 1
                except Exception as e:
                    cache[sid] = None
                    log("study_failed", study=sid, error=repr(e)[:200])
            if i % 25 == 0 or i == len(inputs["ids"]):
                print(f"[step 3] {i}/{len(inputs['ids'])} studies | {time.time() - t0:.0f}s")
        done = max(1, sum(v is not None for v in cache.values()) + sum(v is None for v in cache.values()))
        state["seconds_per_study"] = round((time.time() - t0) / done, 3)

        sub = prior_table(inputs, prior)
        have = [k for k, sid in enumerate(inputs["ids"]) if cache.get(sid) is not None]
        state["n_model"] = len(have)
        state["n_prior_filled"] = len(inputs["ids"]) - len(have)
        if have:
            Xt = np.stack([cache[inputs["ids"][k]] for k in have])
            for t in TARGETS:
                if t in sub.columns and models[t] is not None:
                    sub.loc[have, t] = models[t].predict_proba(Xt)[:, 1]

        problems = validate_submission(sub, inputs["columns"], inputs["ids"])
        if problems:
            state["status"] = "INVALID_USING_PRIOR_ONLY"
            state["notes"] += problems
            print("[step 4] VALIDATION FAILED, keeping the prior-only file:", problems)
        else:
            # Validate a temporary file first; replace the safety-net file only
            # after a successful write and read-back validation.
            tmp_out = inputs["out"].with_name(inputs["out"].name + ".tmp")
            try:
                sub.to_csv(tmp_out, index=False)
                back = pd.read_csv(tmp_out)
                problems = validate_submission(back, inputs["columns"], inputs["ids"])
                if problems:
                    state["status"] = "INVALID_READBACK"
                    state["notes"] += problems
                    tmp_out.unlink(missing_ok=True)
                else:
                    os.replace(tmp_out, inputs["out"])
            except Exception:
                tmp_out.unlink(missing_ok=True)
                raise
            if not problems:
                rate = state["n_prior_filled"] / max(1, state["n_test"])
                if state["n_model"] == 0:
                    state["status"] = "PRIOR_ONLY"
                elif rate > HIGH_FAILURE_RATE:
                    state["status"] = "WARN_HIGH_FAILURE_RATE"
                elif any(e["event"] == "time_budget_exceeded" for e in LOG):
                    state["status"] = "WARN_TIME_BUDGET_EXCEEDED"
                print(f"[step 4] wrote and re-validated {inputs['out']} | model predictions for {state['n_model']}/{state['n_test']} studies")
        state["prediction_summary"] = {
            t: {"mean": round(float(sub[t].mean()), 4), "std": round(float(sub[t].std(ddof=0)), 4)}
            for t in TARGETS if t in sub.columns}
    except Exception:
        state["status"] = "ERROR_PRIOR_ONLY" if inputs is not None else "ERROR_NO_FILE"
        state["notes"].append(traceback.format_exc()[-1500:])
        print("PIPELINE ERROR (prior-only file, if any, left in place):\n", traceback.format_exc())
    finally:
        gc.collect()
    return state

In [4]:
RESULT = run_pipeline()
print("\nstatus:", RESULT["status"])

[step 0] prior-only safety-net file written: /kaggle/working/simulated_submission.csv (100 studies)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[step 1] encoder loaded on cuda
Training features: 58/58 gold studies in 55s
[step 2] fitted 12/12 target models on 58 studies
[step 3] 25/100 studies | 14s
[step 3] 50/100 studies | 25s
[step 3] 75/100 studies | 43s
[step 3] 100/100 studies | 54s
[step 4] wrote and re-validated /kaggle/working/simulated_submission.csv | model predictions for 100/100 studies

status: OK


In [5]:
report = {
    "status": RESULT["status"],
    "simulated_run": bool(RESULT.get("simulated")),
    "n_studies": RESULT["n_test"],
    "n_with_model_predictions": RESULT["n_model"],
    "n_filled_with_prior": RESULT["n_prior_filled"],
    "seconds_per_study_extraction": RESULT["seconds_per_study"],
    "total_elapsed_seconds": round(elapsed(), 1),
    "device": RESULT["device"],
    "series_rung_used_train": dict(RESULT["train_rungs"]),
    "series_rung_used_test": dict(RESULT["test_rungs"]),
    "slice_order_modes_test": dict(RESULT["order_modes"]),
    "slice_range": list(SLICE_RANGE),
    "notes": RESULT["notes"],
    "prediction_summary": RESULT.get("prediction_summary", {}),
    "event_counts": dict(Counter(e["event"] for e in LOG)),
    "first_events": LOG[:60],
}
try:
    (WORK / "run_report.json").write_text(json.dumps(report, indent=2, default=str))
except Exception as e:
    print("could not write run_report.json:", e)

print("=" * 70)
print("STATUS:", report["status"])
print(f"studies: {report['n_studies']} | model predictions: {report['n_with_model_predictions']} | prior-filled: {report['n_filled_with_prior']}")
print("rungs (test):", report["series_rung_used_test"], "| order modes:", report["slice_order_modes_test"])
print("events:", report["event_counts"])
print("seconds per study:", report["seconds_per_study_extraction"], "| total seconds:", report["total_elapsed_seconds"])
if report["simulated_run"]:
    print("\nSIMULATED RUN: no real submission was written. Set SIMULATE_N = 0 before the real run.")
elif report["status"] != "OK":
    print("\n>>> DO NOT SUBMIT YET: status is not OK. Read run_report.json notes and events above. <<<")
else:
    print("\nOK: submission.csv written and validated.")
print("=" * 70)
if report["notes"]:
    print("notes:", report["notes"])

STATUS: OK
studies: 100 | model predictions: 100 | prior-filled: 0
rungs (test): {'sagittal_fluid': 97, 'coronal_fluid': 3} | order modes: {'instance_number': 100}
events: {}
seconds per study: 0.544 | total seconds: 110.2

SIMULATED RUN: no real submission was written. Set SIMULATE_N = 0 before the real run.


## Notes for later
- To change the model, change `build_training_set` / `fit_models` only; `extract_study` is used identically for training and test, which keeps train/test features consistent.
- If you change `SLICE_RANGE`, `N_SLICES` or the preprocessing, re-run the five-fold evaluation first; the numbers in the experiment register are for the settings in this notebook.
- `run_report.json` lists every fallback (series, slice, study). A high count of `series_failed` or `study_failed` is the first thing to investigate.